In [2]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10793358 sha256=7cb5dd37e4eef46a3fc4f0d8e70870fbb31a314cbd9daf3d3284a657a4c464a4
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [16]:
!pip install implicit pandarallel -q

import os, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRanker, Pool
import implicit
from tqdm.auto import tqdm

# Для текстов
from pandarallel import pandarallel
import nltk
import re
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_sw
from nltk.stem import SnowballStemmer

pandarallel.initialize(progress_bar=True, nb_workers=os.cpu_count())

# Конфиг
DATA_DIR    = "/kaggle/input/datasets/artemnazemtsev/nto-ai"
OUTPUT_PATH = "submission.csv"

TOP_K               = 20
CANDIDATES_PER_USER = 300
INCIDENT_DAYS       = 30
HOLDOUT_DAYS        = 30
RANDOM_SEED         = 42

ALS_FACTORS = 64
ALS_ITERS   = 30

print("Библиотеки и конфиг загружены!")

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Библиотеки и конфиг загружены!


In [17]:
def optimize_dtypes(df):
    for col in df.columns:
        if df[col].dtype == 'int64':
            df[col] = df[col].astype(np.int32)
        elif df[col].dtype == 'float64':
            df[col] = df[col].astype(np.float32)
    return df

print("Загрузка данных...")
interactions = pd.read_csv(f"{DATA_DIR}/interactions.csv", parse_dates=["event_ts"])
targets      = pd.read_csv(f"{DATA_DIR}/targets.csv")
editions     = pd.read_csv(f"{DATA_DIR}/editions.csv")
book_genres  = pd.read_csv(f"{DATA_DIR}/book_genres.csv")
users        = pd.read_csv(f"{DATA_DIR}/users.csv")

# Оптимизация памяти
interactions = optimize_dtypes(interactions)
targets      = optimize_dtypes(targets)
editions     = optimize_dtypes(editions)
book_genres  = optimize_dtypes(book_genres)
users        = optimize_dtypes(users)
interactions['event_type'] = interactions['event_type'].astype(np.int8)

T_MAX            = interactions["event_ts"].max()
T_INCIDENT_START = T_MAX - pd.Timedelta(days=INCIDENT_DAYS)
T_TRAIN_SPLIT    = T_INCIDENT_START - pd.Timedelta(days=HOLDOUT_DAYS)

print(f"Interactions: {len(interactions):,}")
print(f"Сплит трейна: {T_TRAIN_SPLIT.date()}, Начало инцидента: {T_INCIDENT_START.date()}")

Загрузка данных...
Interactions: 443,278
Сплит трейна: 2025-10-01, Начало инцидента: 2025-10-31


In [18]:
RU_STOP = set(nltk_sw.words("russian"))
EN_STOP = set(nltk_sw.words("english"))
STOPWORDS = RU_STOP | EN_STOP
STEMMER = SnowballStemmer("russian")

_RE_HTML  = re.compile(r"<[^>]+>")
_RE_CHARS = re.compile(r"[^а-яёa-z\s]")

def stem_and_filter(text: str, max_tokens: int = 200) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    tokens = [STEMMER.stem(t) for t in text.split() if len(t) >= 3 and t not in STOPWORDS]
    return " ".join(tokens[:max_tokens])

def vector_clean(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str)
    s = s.str.normalize("NFKC").str.replace(_RE_HTML, " ", regex=True).str.lower().str.replace(_RE_CHARS, " ", regex=True)
    return s

print("Очистка текстов...")
t_prep = vector_clean(editions["title"])
d_prep = vector_clean(editions["description"])

print("Стемминг (многопоточно)...")
editions["title_clean"] = t_prep.parallel_apply(stem_and_filter)
editions["desc_clean"]  = d_prep.parallel_apply(lambda x: stem_and_filter(x, max_tokens=120))

ed_avg_rating_map = (
    interactions[interactions["event_type"] == 2]
    .groupby("edition_id")["rating"].mean().fillna(0).to_dict()
)

Очистка текстов...
Стемминг (многопоточно)...


In [27]:
def build_features(iact: pd.DataFrame, T_split):
    pos = iact[iact["event_type"].isin([1, 2])].copy()
    
    # 1. Базовые счетчики
    ep = pos.groupby("edition_id").size().rename("edition_pop")
    ep_w = pos[pos["event_type"] == 1].groupby("edition_id").size().rename("wishlist_pop")
    ep_r = pos[pos["event_type"] == 2].groupby("event_type").size().rename("read_pop") # Fix: groupby edition_id
    ep_r = pos[pos["event_type"] == 2].groupby("edition_id").size().rename("read_pop")
    bgc = book_genres.groupby("book_id").size().rename("book_genre_count")
    
    # 2. "Горячая" популярность (последние 14 дней в данном сплите)
    recent_cutoff = T_split - pd.Timedelta(days=14)
    recent_pop = pos[pos["event_ts"] >= recent_cutoff].groupby("edition_id").size().rename("recent_pop")
    
    ed_feat = editions[["edition_id", "book_id", "author_id", "publication_year", "language_id", "title_clean", "desc_clean"]].set_index("edition_id")
    ed_feat = ed_feat.join(ep).join(ep_w).join(ep_r).join(recent_pop).join(bgc, on="book_id").fillna(0)
    ed_feat["pop_log"] = np.log1p(ed_feat["edition_pop"]).astype(np.float32)
    
    # 3. Фичи пользователей + АГРЕГАТЫ ПО ИСТОРИИ
    u_total = pos.groupby("user_id").size().rename("user_activity")
    
    # Сливаем историю с фичами изданий, чтобы понять, ЧТО читает юзер
    hist_stats = pos[["user_id", "edition_id"]].merge(
        ed_feat[["publication_year", "edition_pop", "recent_pop"]].reset_index(), 
        on="edition_id"
    )
    u_hist_agg = hist_stats.groupby("user_id").agg({
        "publication_year": ["mean", "std"],
        "edition_pop": ["mean", "max"],
        "recent_pop": "mean"
    })
    u_hist_agg.columns = ["_".join(c) for c in u_hist_agg.columns]
    
    u_feat = users[["user_id", "gender", "age"]].set_index("user_id")
    u_feat = u_feat.join(u_total).join(u_hist_agg).fillna(0)
    u_feat["user_activity_log"] = np.log1p(u_feat["user_activity"]).astype(np.float32)
    
    # 4. Профили (жанры и авторы)
    df_g = pos.merge(editions[["edition_id", "book_id"]], on="edition_id").merge(book_genres, on="book_id")
    ugp = df_g.groupby(["user_id", "genre_id"]).size().rename("cnt").reset_index()
    ugp["user_genre_affinity"] = (ugp["cnt"] / ugp.groupby("user_id")["cnt"].transform("sum")).astype(np.float32)
    
    df_a = pos.merge(editions[["edition_id", "author_id"]], on="edition_id").dropna(subset=["author_id"])
    uap = df_a.groupby(["user_id", "author_id"]).size().rename("cnt").reset_index()
    uap["user_author_affinity"] = (uap["cnt"] / uap.groupby("user_id")["cnt"].transform("sum")).astype(np.float32)
    
    return ed_feat, u_feat, ugp[["user_id", "genre_id", "user_genre_affinity"]], uap[["user_id", "author_id", "user_author_affinity"]], pos

In [34]:
from implicit.nearest_neighbours import bm25_weight

def train_als_and_get_matrix(pos_iact: pd.DataFrame):
    pos = pos_iact.copy()
    # Веса: чтение важнее вишлиста в 3 раза
    pos["w"] = pos["event_type"].map({1: 1.0, 2: 3.0})

    ue_enc, ei_enc = LabelEncoder(), LabelEncoder()
    u_idx = ue_enc.fit_transform(pos["user_id"])
    e_idx = ei_enc.fit_transform(pos["edition_id"])

    # Создаем матрицу
    user_item_csr = sparse.csr_matrix(
        (pos["w"].values, (u_idx, e_idx)), 
        shape=(len(ue_enc.classes_), len(ei_enc.classes_))
    )
    
    # ЖЕСТКИЙ МЕТОД №1: BM25 Weighting перед ALS
    # Это заставляет модель обращать внимание на редкие книги, а не только на хиты
    user_item_csr = bm25_weight(user_item_csr, K1=100, B=0.8).tocsr()

    als = implicit.als.AlternatingLeastSquares(
        factors=128, # Увеличили размерность для более тонких связей
        iterations=40, 
        regularization=0.01,
        use_gpu=True, 
        random_state=RANDOM_SEED
    )
    als.fit(user_item_csr) 

    U = als.user_factors.to_numpy().astype(np.float32)
    V = als.item_factors.to_numpy().astype(np.float32)
    
    # Центроиды для I2I (расстояние до среднего вектора интересов)
    U_hist_centroid = (user_item_csr.dot(V) / np.array(user_item_csr.getnnz(axis=1)).reshape(-1, 1).clip(min=1)).astype(np.float32)
    
    U_norm = U / np.linalg.norm(U, axis=1, keepdims=True).clip(min=1e-9)
    V_norm = V / np.linalg.norm(V, axis=1, keepdims=True).clip(min=1e-9)
    U_hist_norm = U_hist_centroid / np.linalg.norm(U_hist_centroid, axis=1, keepdims=True).clip(min=1e-9)

    u2als = {uid: i for i, uid in enumerate(ue_enc.classes_)}
    e2als = {eid: i for i, eid in enumerate(ei_enc.classes_)}
    
    return als, user_item_csr, U, V, U_norm, V_norm, U_hist_norm, u2als, e2als, ue_enc.classes_, ei_enc.classes_

In [29]:
def generate_candidates(targets_list, als_model, user_item_csr, u2als, e2als_arr, pop_items, ed_feat, ugp, uap, k=150):
    print("Генерация ALS U2I...")
    cands_list = []
    batch_size = 2000
    
    # 1. ALS Candidates
    for i in tqdm(range(0, len(targets_list), batch_size)):
        batch_users = targets_list[i:i+batch_size]
        u_indices = [u2als.get(u, -1) for u in batch_users]
        
        valid_mask = np.array(u_indices) != -1
        v_users = np.array(batch_users)[valid_mask]
        v_indices = np.array(u_indices)[valid_mask]
        
        if len(v_indices) == 0: continue
            
        ids, scores = als_model.recommend(v_indices, user_item_csr[v_indices], N=k, filter_already_liked_items=True)
        
        for idx, uid in enumerate(v_users):
            valid_items = ids[idx][ids[idx] != -1]
            eids = e2als_arr[valid_items]
            scrs = scores[idx][ids[idx] != -1]
            cands_list.append(pd.DataFrame({"user_id": uid, "edition_id": eids, "cand_score": scrs, "src_als": 1}))
            
    cands_df = pd.concat(cands_list, ignore_index=True) if cands_list else pd.DataFrame(columns=["user_id", "edition_id", "cand_score", "src_als"])
    
    # 2. Добивка популярными для холодных юзеров
    missing_users = set(targets_list) - set(cands_df['user_id'].unique())
    if missing_users:
        pop_cands = pd.DataFrame({
            'user_id': np.repeat(list(missing_users), len(pop_items)),
            'edition_id': np.tile(pop_items, len(missing_users)),
            'cand_score': 0.0, 'src_als': 0
        })
        cands_df = pd.concat([cands_df, pop_cands], ignore_index=True)

    return cands_df.drop_duplicates(subset=["user_id", "edition_id"])

In [35]:
def build_ranking_dataset(cands_df, u_feat, ed_feat, ugp, uap, U, V, U_norm, V_norm, U_hist_norm, u2als, e2als):
    # Базовый джоин
    df = cands_df.merge(u_feat, on='user_id', how='left')
    df = df.merge(ed_feat, on='edition_id', how='left')
    
    # ЖЕСТКИЙ МЕТОД №2: Cross-Features (Относительные признаки)
    # Помогают CatBoost понять "подходит ли эта книга ЭТОМУ юзеру"
    df["diff_pub_year"] = df["publication_year"] - df["publication_year_mean"]
    df["rel_popularity"] = df["edition_pop"] / (df["edition_pop_mean"] + 1e-6)
    df["rel_recent_pop"] = df["recent_pop"] / (df["recent_pop_mean"] + 1e-6)
    
    # ALS компоненты и расстояния
    u_idxs = np.where(df["user_id"].map(u2als).notna(), df["user_id"].map(u2als).fillna(0).astype(int), -1)
    e_idxs = np.where(df["edition_id"].map(e2als).notna(), df["edition_id"].map(e2als).fillna(0).astype(int), -1)
    valid = (u_idxs != -1) & (e_idxs != -1)
    
    df["als_dot"] = 0.0
    df["als_cos"] = 0.0
    df["als_centroid_cos"] = 0.0
    
    if valid.any():
        u_v, e_v = u_idxs[valid], e_idxs[valid]
        df.loc[valid, "als_dot"] = (U[u_v] * V[e_v]).sum(axis=1)
        df.loc[valid, "als_cos"] = (U_norm[u_v] * V_norm[e_v]).sum(axis=1)
        df.loc[valid, "als_centroid_cos"] = (U_hist_norm[u_v] * V_norm[e_v]).sum(axis=1)
        
        # Добавляем ТОП-16 компонент эмбеддингов
        for i in range(16):
            df[f"u_e_{i}"] = 0.0
            df[f"i_e_{i}"] = 0.0
            df.loc[valid, f"u_e_{i}"] = U[u_v, i]
            df.loc[valid, f"i_e_{i}"] = V[e_v, i]

    # Мягкое заполнение
    df = df.fillna(0)
    # Удаляем лишнее для экономии памяти
    gc.collect()
    return df

In [36]:
print("--- TEMPORAL SPLIT ---")
fake_history = interactions[interactions["event_ts"] < T_TRAIN_SPLIT].copy()
fake_holdout = interactions[(interactions["event_ts"] >= T_TRAIN_SPLIT) & (interactions["event_type"].isin([1, 2]))].copy()

train_users = list(set(fake_history["user_id"]) & set(fake_holdout["user_id"]))
pos_holdout_pairs = set(zip(fake_holdout["user_id"], fake_holdout["edition_id"]))

ed_feat_tr, u_feat_tr, ugp_tr, uap_tr, pos_tr = build_features(fake_history, T_TRAIN_SPLIT)
als_tr, csr_tr, U_tr, V_tr, Un_tr, Vn_tr, Uhist_tr, u2als_tr, e2als_tr, u_arr_tr, e_arr_tr = train_als_and_get_matrix(pos_tr)


pop_items_tr = fake_history['edition_id'].value_counts().head(50).index.tolist()

train_cands = generate_candidates(train_users, als_tr, csr_tr, u2als_tr, e_arr_tr, pop_items_tr, ed_feat_tr, ugp_tr, uap_tr)

# Разметка таргета
train_cands['target'] = train_cands.apply(lambda x: 1.0 if (x['user_id'], x['edition_id']) in pos_holdout_pairs else 0.0, axis=1)

train_df = build_ranking_dataset(train_cands, u_feat_tr, ed_feat_tr, ugp_tr, uap_tr, U_tr, V_tr, Un_tr, Vn_tr, Uhist_tr, u2als_tr, e2als_tr)

train_df = train_df.sort_values('user_id').reset_index(drop=True)
y_train = train_df.pop('target').values
groups_train = train_df['user_id'].values

--- TEMPORAL SPLIT ---


  0%|          | 0/40 [00:00<?, ?it/s]

Генерация ALS U2I...


  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import gc
gc.collect()

In [37]:
def train_ensemble(df, feat_cols, text_cols, n_models=3):
    models = []
    
    # Общие параметры для всех моделей в ансамбле
    # YetiRank — это стандарт индустрии для NDCG
    base_params = {
        "loss_function": "YetiRank",
        "iterations": 1500, # Увеличили итерации
        "learning_rate": 0.03,
        "depth": 6,
        "task_type": "GPU",
        "text_processing": {
            "tokenizers": [{"tokenizer_id": "Space", "delimiter": " ", "lowercasing": "true"}],
            "dictionaries": [{"dictionary_id": "Word", "max_dictionary_size": "50000", "gram_order": "1"}],
            "feature_processing": {"default": [{"dictionaries_names": ["Word"], "feature_calcers": ["BoW"], "tokenizers_names": ["Space"]}]}
        },
        "verbose": 100,
        "early_stopping_rounds": 100
    }

    # Создаем Pool один раз для экономии памяти
    train_pool = Pool(
        data=df[feat_cols + text_cols],
        label=y_train,
        group_id=groups_train,
        text_features=text_cols
    )

    print(f"Начинаю обучение ансамбля из {n_models} моделей...")
    
    for i in range(n_models):
        print(f"\n>>> Обучение модели #{i+1}...")
        curr_params = base_params.copy()
        curr_params["random_seed"] = RANDOM_SEED + i # Каждая модель видит данные под своим углом
        
        m = CatBoostRanker(**curr_params)
        m.fit(train_pool)
        models.append(m)
        
    return models

# 1. Определяем колонки
text_cols = ["title_clean", "desc_clean"]
drop_cols = ["user_id", "edition_id", "book_id", "author_id"]
feat_cols = [c for c in train_df.columns if c not in drop_cols + text_cols]

# 2. Запускаем обучение
ensemble_models = train_ensemble(train_df, feat_cols, text_cols, n_models=3)

Начинаю обучение ансамбля из 3 моделей...

>>> Обучение модели #1...


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 72.1ms	remaining: 1m 48s
100:	total: 3.67s	remaining: 50.9s
200:	total: 7.2s	remaining: 46.6s
300:	total: 10.7s	remaining: 42.7s
400:	total: 14.3s	remaining: 39.1s
500:	total: 17.8s	remaining: 35.5s
600:	total: 21.3s	remaining: 31.9s
700:	total: 24.7s	remaining: 28.1s
800:	total: 28s	remaining: 24.4s
900:	total: 31.3s	remaining: 20.8s
1000:	total: 34.6s	remaining: 17.2s
1100:	total: 37.8s	remaining: 13.7s
1200:	total: 41s	remaining: 10.2s
1300:	total: 44.2s	remaining: 6.76s
1400:	total: 47.4s	remaining: 3.35s
1499:	total: 50.5s	remaining: 0us

>>> Обучение модели #2...


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 65.6ms	remaining: 1m 38s
100:	total: 3.58s	remaining: 49.5s
200:	total: 6.97s	remaining: 45.1s
300:	total: 10.3s	remaining: 41s
400:	total: 13.7s	remaining: 37.6s
500:	total: 17.1s	remaining: 34s
600:	total: 20.4s	remaining: 30.6s
700:	total: 23.8s	remaining: 27.1s
800:	total: 27.2s	remaining: 23.7s
900:	total: 30.5s	remaining: 20.3s
1000:	total: 33.8s	remaining: 16.9s
1100:	total: 37.2s	remaining: 13.5s
1200:	total: 40.4s	remaining: 10.1s
1300:	total: 43.7s	remaining: 6.68s
1400:	total: 46.9s	remaining: 3.31s
1499:	total: 50s	remaining: 0us

>>> Обучение модели #3...


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 66.3ms	remaining: 1m 39s
100:	total: 3.62s	remaining: 50.2s
200:	total: 7.02s	remaining: 45.3s
300:	total: 10.4s	remaining: 41.5s
400:	total: 13.8s	remaining: 37.8s
500:	total: 17.1s	remaining: 34s
600:	total: 20.3s	remaining: 30.4s
700:	total: 23.7s	remaining: 27s
800:	total: 27s	remaining: 23.5s
900:	total: 30.2s	remaining: 20.1s
1000:	total: 33.6s	remaining: 16.7s
1100:	total: 36.8s	remaining: 13.3s
1200:	total: 40.1s	remaining: 9.98s
1300:	total: 43.3s	remaining: 6.63s
1400:	total: 46.5s	remaining: 3.28s
1499:	total: 49.6s	remaining: 0us


In [39]:
text_cols = ["title_clean", "desc_clean"]
drop_cols = ["user_id", "edition_id", "book_id", "author_id"]
feat_cols = [c for c in train_df.columns if c not in drop_cols + text_cols]

pool = Pool(
    data=train_df[feat_cols + text_cols],
    label=y_train,
    group_id=groups_train,
    text_features=text_cols
)

model = CatBoostRanker(
    loss_function="YetiRank",
    iterations=800,
    learning_rate=0.05,
    depth=6,
    task_type="GPU",
    text_processing={
        "tokenizers": [{"tokenizer_id": "Space", "delimiter": " ", "lowercasing": "true"}],
        "dictionaries": [{"dictionary_id": "Word", "max_dictionary_size": "50000", "gram_order": "1"}],
        "feature_processing": {"default": [{"dictionaries_names": ["Word"], "feature_calcers": ["BoW"], "tokenizers_names": ["Space"]}]}
    },
    random_seed=RANDOM_SEED,
    verbose=100
)

print("Обучение CatBoost...")
model.fit(pool)

Обучение CatBoost...


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 68.7ms	remaining: 54.9s
100:	total: 3.7s	remaining: 25.6s
200:	total: 7.16s	remaining: 21.3s
300:	total: 10.6s	remaining: 17.6s
400:	total: 14.1s	remaining: 14s
500:	total: 17.5s	remaining: 10.5s
600:	total: 20.9s	remaining: 6.92s
700:	total: 24.2s	remaining: 3.41s
799:	total: 27.4s	remaining: 0us


In [40]:
print("--- [STAGE 1] INFERENCE PREPARATION ---")
# 1. Генерируем фичи на полном логе перед окном инцидента
# (Используем pos_f, который возвращается из обновленной функции build_features)
ed_feat_f, u_feat_f, ugp_f, uap_f, pos_f = build_features(interactions, T_INCIDENT_START)

# 2. Переобучаем ALS на полном логе (с BM25 весами)
als_f, csr_f, U_f, V_f, Un_f, Vn_f, Uhist_f, u2als_f, e2als_f, u_arr_f, e_arr_f = train_als_and_get_matrix(pos_f)

# 3. Настраиваем воронку кандидатов
CANDIDATES_PER_USER = 300 # Расширяем воронку для CatBoost
pop_items_f = interactions['edition_id'].value_counts().head(100).index.tolist()
test_users = targets["user_id"].unique().tolist()

print(f"Генерация {CANDIDATES_PER_USER} кандидатов для {len(test_users)} пользователей...")
test_cands = generate_candidates(test_users, als_f, csr_f, u2als_f, e_arr_f, pop_items_f, ed_feat_f, ugp_f, uap_f, k=CANDIDATES_PER_USER)

print("Сборка датасета для ранжирования...")
test_df = build_ranking_dataset(
    test_cands, u_feat_f, ed_feat_f, ugp_f, uap_f, 
    U_f, V_f, Un_f, Vn_f, Uhist_f, u2als_f, e2als_f
)

# --- [STAGE 2] ENSEMBLE PREDICT ---
print(f"Инференс ансамбля из {len(ensemble_models)} моделей...")
test_pool = Pool(data=test_df[feat_cols + text_cols], text_features=text_cols)

all_preds = []
for i, m in enumerate(ensemble_models):
    print(f"Предсказание модели #{i+1}...")
    all_preds.append(m.predict(test_pool))

# Усредняем предсказания
test_df["score"] = np.mean(all_preds, axis=0)

# --- [STAGE 3] POST-PROCESSING & BLENDING ---
# ЖЕСТКИЙ МЕТОД: Добавляем небольшой вес за "горячую" популярность
# Это помогает вытащить потеряшки, которые были в тренде в момент сбоя
test_df["score"] = test_df["score"] + 0.07 * np.log1p(test_df["recent_pop"])

# Ранжируем
res = test_df[["user_id", "edition_id", "score"]].sort_values(["user_id", "score"], ascending=[True, False])
res = res.groupby("user_id").head(TOP_K).copy()

# --- [STAGE 4] FILLING GAPS & EXPORT ---
print("Формирование финального списка и проверка на пропуски...")
per_user = {u: g["edition_id"].tolist() for u, g in res.groupby("user_id")}
final_rows = []

for uid in tqdm(test_users):
    seen = per_user.get(uid, [])
    
    # Добавляем предсказанные
    for e in seen:
        final_rows.append({"user_id": uid, "edition_id": e})
    
    # Добивка популярными, если кандидатов не хватило (холодные юзеры)
    need = TOP_K - len(seen)
    if need > 0:
        extra = [e for e in pop_items_f if e not in seen][:need]
        for e in extra:
            final_rows.append({"user_id": uid, "edition_id": e})

# Создаем сабмит
submission = pd.DataFrame(final_rows)
submission["rank"] = submission.groupby("user_id").cumcount() + 1

# Финальная проверка структуры (должно быть ровно len(targets) * 20 строк)
expected_len = len(targets) * TOP_K
if len(submission) != expected_len:
    print(f"ВНИМАНИЕ: Ошибка длины! Ожидалось {expected_len}, получено {len(submission)}")

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Успех! Файл '{OUTPUT_PATH}' готов к отправке.")

--- [STAGE 1] INFERENCE PREPARATION ---


  0%|          | 0/40 [00:00<?, ?it/s]

Генерация 300 кандидатов для 3862 пользователей...
Генерация ALS U2I...


  0%|          | 0/2 [00:00<?, ?it/s]

Сборка датасета для ранжирования...
Инференс ансамбля из 3 моделей...
Предсказание модели #1...
Предсказание модели #2...
Предсказание модели #3...
Формирование финального списка и проверка на пропуски...


  0%|          | 0/3862 [00:00<?, ?it/s]

Успех! Файл 'submission.csv' готов к отправке.
